# 利用记忆与压缩构建可靠的智能体

本 Cookbook 展示如何使用 OpenAI Agents SDK，为一个合成的合规调查场景构建证据审阅智能体。

你会先从一个简单的沙箱智能体开始，然后加入两个可靠性原语：

- **压缩（Compaction）**：在上下文窗口有限的前提下支持长时间运行的对话，它会把后续轮次所需的状态带到下一轮，同时缩小上下文体积。
- **记忆（Memory）**：让之后的沙箱智能体运行能复用先前运行中沉淀的工作流经验，而无需重放之前的每一轮对话。

这套可靠性模式很直接：压缩帮助当前这次运行继续下去，记忆帮助后续运行一开始就带着有用的工作流指引，而生成的备忘录始终是由人工审阅的、调查结论的唯一事实来源。

参考资料：

- [OpenAI Agents SDK](https://openai.github.io/openai-agents-python/)
- [Agents SDK sessions](https://openai.github.io/openai-agents-python/sessions/)
- [Agents SDK sandboxing](https://openai.github.io/openai-agents-python/ref/sandbox/)

## 使用场景：合规调查中的证据审阅智能体

合规团队正在调查一项供应商例外审批是否遵守了内部政策。证据以一小批文件的形式陆续到来：政策条文、例外说明、审计观察、审批记录和整改计划。

智能体的职责不是成为调查记录本身。它的职责是帮助审阅人逐步过一遍证据，追踪哪些内容发生了变化，并写出一份简明的备忘录，把有据可查的发现与悬而未决的问题分开。

这个例子之所以适合演示记忆与压缩，是因为这类调查具备真实工作中常见的三个特征：

- **记录会随时间变化。** 后续文件可能缩小或推翻早先的假设。
- **对话可能长时间持续。** 审阅人可能追问、要求修改，并在之后回到同一项工作上。
- **最终产物需要可溯源。** 备忘录应引用证据并保留不确定性，而不是把审阅压平成一个自信却缺乏支撑的结论。

## 这个模式适用于哪里

虽然本 notebook 以合规审阅为例，但同样的模式适用于任何知识工作者需要审阅不断演进的上下文、并产出可供人工审计的产物的场景。

典型的适用场景包括：

- 客服团队把新的政策更新应用到待处理的升级工单上。
- 安全团队审阅事件证据并撰写事件摘要。
- 财务团队在政策、审批和审计记录之间核对例外事项。
- 产品团队在新发布或新模型上线后更新竞争定位。
- 法务或采购团队审阅合同、邮件和审批历史。
- 并购团队消化新的业务规则、操作流程和尽调记录。

在每种情况下，压缩让活跃的审阅在上下文增长时仍能继续，记忆把可复用的工作流经验带到后续运行，而最终产物始终是经过审阅的输出。

## 你将构建什么

使用场景是一次合规证据审阅。团队会陆续收到政策文件、例外说明、审计发现、审批记录和整改计划。智能体帮助审阅证据、保留不确定性，并产出一份带引用的简明备忘录。

你将构建：

1. 一个合成的证据工作区，包含目录结构、清单（manifest）和输出目录。
2. 一个能检查文件并撰写备忘录的简单 `SandboxAgent`。
3. 面向长时间运行工作的压缩检查点。
4. 用于生成可复用工作流经验的 SDK 记忆。
5. 一次把沙箱工具、压缩、记忆和生成产物组合在一起的完整运行。

由于 `RUN_AGENT` 默认是 `False`，你可以在不调用模型的情况下阅读整个 notebook。只有在想执行真实的沙箱工作流时，才把 `RUN_AGENT` 设为 `True`。

### 目录

- [使用场景](#使用场景合规调查中的证据审阅智能体)
- [这个模式适用于哪里](#这个模式适用于哪里)
- [前置条件](#前置条件)
- [环境准备](#环境准备)
- [本 notebook 如何使用 Agents SDK](#本-notebook-如何使用-agents-sdk)
- [记忆与压缩的区别](#记忆与压缩的区别)
- [目录结构与清单](#目录结构与清单)
- [准备一个小型证据工作区](#准备一个小型证据工作区)
- [第 1 步：从简单的智能体配置开始](#第-1-步从简单的智能体配置开始)
- [第 2 步：加入压缩](#第-2-步加入压缩)
- [第 3 步：挂载记忆](#第-3-步挂载记忆)
- [第 4 步：同时启用压缩与记忆运行](#第-4-步同时启用压缩与记忆运行)
- [检查生成的产物](#检查生成的产物)

## 前置条件

要运行真实的智能体工作流，你需要：

- Python 3.10 或更高版本。
- `openai-agents` 包。
- 通过环境变量 `OPENAI_API_KEY` 提供的 OpenAI API key。
- 一个本地的类 Unix 环境，供 `UnixLocalSandboxClient` 使用。本 notebook 使用由沙箱 `Manifest` 创建的合成文件，因此不需要任何外部数据集。

由于 `RUN_AGENT` 默认是 `False`，没有凭据也可以安全地阅读本 notebook。只有在想执行由模型驱动的沙箱运行时，才把 `RUN_AGENT` 设为 `True`。


In [ ]:
# Install or upgrade the OpenAI Agents SDK.
%pip install --upgrade openai-agents

## 环境准备

本 notebook 会把所有合成文件写到 `examples/agents_sdk/.tmp/evidence_review_memory_compaction/` 下，不需要外部数据。

默认关闭追踪（tracing），因为部分组织启用了零数据保留（Zero Data Retention，ZDR），追踪数据的上报可能被拦截。对于合成数据或非 ZDR 环境，你可以设置 `ENABLE_TRACING = True`，在开发时查看追踪记录。

In [ ]:
from __future__ import annotations

import json
import os
import textwrap
from pathlib import Path

RUN_AGENT = False
MODEL = "gpt-5.5"
COMPACTION_MODEL = "gpt-5.4-mini"
WORKFLOW_NAME = "evidence-review-memory-compaction"
FORCE_COMPACTION_CHECKPOINT = True
DISABLE_TRACING = True

if DISABLE_TRACING:
    os.environ["OPENAI_AGENTS_DISABLE_TRACING"] = "true"
else:
    os.environ.pop("OPENAI_AGENTS_DISABLE_TRACING", None)

if RUN_AGENT and not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("Set OPENAI_API_KEY before running the live sandbox workflow.")

print({
    "run_agent": RUN_AGENT,
    "model": MODEL,
    "compaction_model": COMPACTION_MODEL,
    "force_compaction_checkpoint": FORCE_COMPACTION_CHECKPOINT,
    "tracing_disabled": DISABLE_TRACING,
})


## 本 notebook 如何使用 Agents SDK

**沙箱智能体（sandbox agent）**是在受控工作区中运行的 Agents SDK 智能体。在本 notebook 中，这个工作区包含合成的证据文件、一个 `manifest.csv`、一个输出目录，以及由 SDK 生成的记忆文件。

沙箱给智能体提供了一个有边界的地方来检查文件和写入产物。应用不再把每份文档都贴进提示词，而是创建一个工作区，让智能体使用以下能力：

- `Filesystem()`：读写工作区文件。
- `Shell()`：列出文件、查看文档、跨批次搜索。
- `Compaction()`：当活跃上下文增长时，支持长时间运行的审阅。
- `Memory()`：为后续沙箱智能体运行存储可复用的工作流经验。

备忘录始终是由人工审阅的产物。工具帮助智能体工作，压缩帮助它继续，记忆帮助后续运行改进，生成的产物承载可供审阅的输出。

## 记忆与压缩的区别

区分这两个概念的一个好办法，是问清楚各自允许把什么带到后面。

| 问题 | 压缩（Compaction） | 记忆（Memory） |
|---|---|---|
| 它解决什么问题？ | 在上下文增长时让单次长时间运行得以继续。 | 用可复用的工作流经验改进后续运行。 |
| 它总结什么？ | 活跃对话和工作状态。 | 值得复用的模式、偏好和流程经验。 |
| 它应该存储调查结论吗？ | 不。它可以保留工作状态，但备忘录才是经审阅的产物。 | 不。存储工作流经验，而不是案件相关的事实。 |
| 什么时候有用？ | 审阅进行中，尤其是在后续批次或追问轮次之前。 | 在对相似证据工作流的多次审阅之间。 |

在本 notebook 中，合规备忘录是调查输出的事实来源。记忆被有意限定在审阅人偏好和工作流习惯上，例如先看清单、保留不确定性、让被推翻的假设保持可见。

## 目录结构与清单

智能体基于一个小型文件工作区工作。目录结构刻意保持简单：证据文件按批次分组，生成的输出放在 `outputs/` 下，清单则为智能体提供一张可用文档的紧凑地图。

### Manifest 功能

在 Agents SDK 中，`Manifest` 是沙箱智能体新会话的工作区契约。它描述了新沙箱会话启动时应当存在的文件、目录、挂载、环境变量、用户、用户组及相关工作区配置。

本地 SDK 实现定义了以下核心字段：

| Manifest 字段 | 控制什么 | 在本 Cookbook 中如何使用 |
|---|---|---|
| `root` | 工作区根路径，默认为 `/workspace`。 | 除非沙箱提供方要求不同的根路径，否则保持默认。 |
| `entries` | 需要物化的文件、目录、本地文件、本地目录、仓库或挂载。 | 把 `README.md`、`manifest.csv`、输入文档和 `outputs/` 放在这里。 |
| `environment` | 沙箱启动时可用的环境变量。 | 仅用于非机密的运行时配置。凭据不要出现在提示词和已提交的 notebook 中。 |
| `users` / `groups` | 沙箱本地的操作系统账户和用户组，适用于支持它们的提供方。 | Cookbook 里通常用不到，对生产环境隔离有用。 |
| `extra_path_grants` | 额外的路径授权，对 Unix 本地工作流尤其有用。 | 仅在沙箱需要对宿主机路径做受限读写时谨慎使用。 |
| `remote_mount_command_allowlist` | 允许对远程挂载执行的命令。 | 挂载外部存储或数据室时保持窄范围。 |

Manifest 条目路径应相对于工作区。避免绝对路径和 `..` 逃逸，这样同一个智能体才能在 Unix 本地、Docker 和托管沙箱提供方之间迁移。


### 目录与清单的最佳实践

- 把源文档、清单、辅助文件和输出目录放进 `Manifest`，而不是把大段内容贴进提示词。
- 把较长的任务说明放在 `README.md`、`task.md` 或 `AGENTS.md` 等工作区文件中；智能体指令只聚焦于行为和边界。
- 使用稳定的文档 ID 和机器可读的清单文件，这样生成的备忘录可以引用来源，审阅人也能顺着路径回溯到证据。
- 让 `Memory()` 管理自己的记忆产物。默认情况下，沙箱记忆使用工作区下的 `memories/` 和 `sessions/`。
- 把生成的产物放在 `outputs/` 下，便于应用在运行后检查、复制、校验或归档。
- 保持挂载范围窄。如果挂载数据室，只挂载智能体应该读写的部分。
- 把机密当作由应用或沙箱提供方注入的运行时配置，而不是提示词文本或已提交的清单内容。
- 教程中优先使用小型合成的 `File(...)` 或 `Dir(...)` 条目，生产规模的数据集再换成 `LocalDir`、`GitRepo` 或存储挂载。


In [ ]:
# This is a visual preview of the sandbox workspace structure.
# The next cell builds the actual Manifest entries manually.
WORKSPACE_TREE = """
/workspace/
  README.md
  manifest.csv
  docs/
    batch_1/
    batch_2/
    batch_3/
  outputs/
  memories/    # Generated by Memory()
  sessions/    # Generated by Memory()
""".strip()

print(WORKSPACE_TREE)


## 准备一个小型证据工作区

`Manifest` 描述了新沙箱工作区的初始文件。本教程的工作区包含：

- 一个按批次和文档 ID 列出文档的 `manifest.csv`，
- 三个小型文档批次，
- 一个用于存放审阅备忘录的输出目录。

我们之后挂载的唯一记忆原语是 SDK 的 `Memory()` 能力。调查发现保留在生成的审阅备忘录中，在那里可以被引用和检查。


In [ ]:
from agents.sandbox import Manifest
from agents.sandbox.entries import Dir, File


def workspace_file(text: str) -> File:
    return File(content=textwrap.dedent(text).strip().encode("utf-8") + b"\n")


def build_evidence_manifest() -> Manifest:
    return Manifest(
        entries={
            "README.md": workspace_file(
                """
                # Evidence Review Workspace

                Review the documents in batch order. Cite document IDs from
                `manifest.csv` when making findings. Write the final memo to
                `outputs/compliance_review_memo.md`.
                """
            ),
            "manifest.csv": workspace_file(
                """
                doc_id,batch,path,description
                ACME-B1-001,1,docs/batch_1/payment_policy.txt,Baseline payment policy
                ACME-B1-002,1,docs/batch_1/vendor_exception.txt,Vendor exception note
                ACME-B2-001,2,docs/batch_2/audit_followup.txt,Audit follow-up request
                ACME-B2-002,2,docs/batch_2/approval_thread.txt,Approval clarification
                ACME-B3-001,3,docs/batch_3/remediation_plan.txt,Remediation plan
                """
            ),
            "docs/batch_1/payment_policy.txt": workspace_file(
                """
                doc_id: ACME-B1-001
                ACME requires two approvals for payments over $50,000. Exceptions must
                be logged with Finance Ops and reviewed within five business days.
                """
            ),
            "docs/batch_1/vendor_exception.txt": workspace_file(
                """
                doc_id: ACME-B1-002
                A vendor onboarding exception was approved verbally for Northwind
                Logistics because the renewal was time-sensitive. The note does not show
                a Finance Ops log entry.
                """
            ),
            "docs/batch_2/audit_followup.txt": workspace_file(
                """
                doc_id: ACME-B2-001
                Internal Audit asked Finance Ops to confirm whether Northwind Logistics
                received post-approval review. The request says missing exception logs
                should be treated as a control gap until resolved.
                """
            ),
            "docs/batch_2/approval_thread.txt": workspace_file(
                """
                doc_id: ACME-B2-002
                The approval thread says Legal approved the vendor exception, but Finance
                Ops approval was still pending when the payment was released.
                """
            ),
            "docs/batch_3/remediation_plan.txt": workspace_file(
                """
                doc_id: ACME-B3-001
                The remediation plan requires Finance Ops to reconcile all verbal vendor
                exceptions from Q4 and add retrospective control attestations.
                """
            ),
            "outputs": Dir(),
        }
    )


manifest = build_evidence_manifest()
print(f"Workspace entries: {len(manifest.entries)}")


## 第 1 步：从简单的智能体配置开始

首先，构建一个没有记忆和压缩的智能体。目标是在加入原语之前先把基线行为弄清楚。

一个细节：`SandboxAgent` 的默认值可能包含内置能力。为了让基线保持明确，请显式传入你想要的能力列表。这里只包含智能体检查文件和写入产物所需的工作区工具：`Filesystem()` 和 `Shell()`。我们有意**不**在此时挂载 `Compaction()` 或 `Memory()`。

- `Filesystem()` 为沙箱智能体提供面向文件的工作区访问，使其可以读取已准备好的证据并写入备忘录产物。[Sandbox Agents 指南](https://developers.openai.com/api/docs/guides/agents/sandboxes#give-the-agent-capabilities)把能力（capabilities）描述为给 `SandboxAgent` 挂载沙箱原生行为与工具的方式。
- `Shell()` 让智能体通过终端命令检查工作区，例如列出文件、打开证据文档、跨批次搜索关键词。Sandbox Agents 指南提到 `Shell()` 是默认能力之一，[Shell 工具指南](https://developers.openai.com/api/docs/guides/tools-shell)则说明 shell 为模型提供了用于托管或本地执行的终端环境。
- 对于这个基线，这两个能力就够了：`Filesystem()` 负责工作区读写，`Shell()` 负责确定性的检查和搜索。记忆和压缩只在基线框架清晰之后才加入。


In [ ]:
from agents.sandbox import SandboxAgent
from agents.sandbox.capabilities import Filesystem, Shell

BASELINE_INSTRUCTIONS = """
You are an evidence review agent for a compliance investigation.

Review documents in batch order. Keep these boundaries clear:
- Cite document IDs from `manifest.csv` for each finding.
- If evidence is incomplete, record an open question instead of guessing.
- Write a concise reviewer memo to `outputs/compliance_review_memo.md`.
- Use the generated memo as the reviewer-facing investigation artifact.
""".strip()


def build_baseline_agent() -> SandboxAgent:
    return SandboxAgent(
        name="Evidence Review Agent",
        model=MODEL,
        instructions=BASELINE_INSTRUCTIONS,
        default_manifest=build_evidence_manifest(),
        capabilities=[
            Filesystem(),
            Shell(),
        ],
    )


baseline_agent = build_baseline_agent()
print([type(capability).__name__ for capability in baseline_agent.capabilities])


In [ ]:
from agents import Runner
from agents.run import RunConfig
from agents.sandbox import SandboxRunConfig
from agents.sandbox.sandboxes.unix_local import UnixLocalSandboxClient

BASELINE_TASK = """
Review Batch 1 only, then draft `outputs/compliance_review_memo.md` for a
compliance reviewer. Include cited findings and open questions.
""".strip()


async def _read_text_file(session, path: str) -> str | None:
    try:
        handle = await session.read(Path(path))
    except Exception as exc:
        if "NotFound" not in type(exc).__name__:
            raise
        return None
    try:
        return handle.read().decode("utf-8", errors="replace")
    finally:
        handle.close()


async def _list_workspace_files(session) -> str:
    result = await session.exec(
        "find outputs memories sessions -maxdepth 4 -type f 2>/dev/null | sort || true",
        timeout=30,
    )
    return result.stdout.decode("utf-8", errors="replace").strip()


async def _read_memory_artifacts(session) -> dict[str, str]:
    memory_artifacts = {}
    for path in ["memories/MEMORY.md", "memories/memory_summary.md"]:
        text = await _read_text_file(session, path)
        if text:
            memory_artifacts[path] = text
    return memory_artifacts


async def run_in_unix_sandbox(agent: SandboxAgent, task: str, *, sdk_session=None) -> dict[str, object]:
    client = UnixLocalSandboxClient()
    session = await client.create(manifest=agent.default_manifest)
    try:
        await session.start()
        result = await Runner.run(
            agent,
            task,
            max_turns=12,
            run_config=RunConfig(
                sandbox=SandboxRunConfig(session=session),
                workflow_name=WORKFLOW_NAME,
                tracing_disabled=DISABLE_TRACING,
            ),
            session=sdk_session,
        )

        compaction_checkpoint = None
        if sdk_session is not None and FORCE_COMPACTION_CHECKPOINT:
            compaction_checkpoint = await force_compaction_checkpoint(sdk_session)

        # Memory generation runs as a sandbox pre-stop hook. Flush it before reading artifacts
        # so `memories/MEMORY.md` and `memories/memory_summary.md` are available here.
        await session.run_pre_stop_hooks()

        memo = await _read_text_file(session, "outputs/compliance_review_memo.md")
        files = await _list_workspace_files(session)
        memory_artifacts = await _read_memory_artifacts(session)

        return {
            "result": result,
            "final_output": str(result.final_output),
            "memo": memo,
            "workspace_files": files,
            "memory_artifacts": memory_artifacts,
            "compaction_checkpoint": compaction_checkpoint,
        }
    finally:
        await session.aclose()


if RUN_AGENT:
    baseline_run = await run_in_unix_sandbox(baseline_agent, BASELINE_TASK)
    print(baseline_run["final_output"])
    print("\n----- END AGENT OUTPUT -----")
else:
    print("RUN_AGENT is False. Baseline agent is configured but not executed.")


## 第 2 步：加入压缩

压缩面向长时间运行的工作。随着对话增长，压缩在保留后续轮次所需状态的同时缩小上下文体积。有三种有用的思考方式：

1. **使用 `Compaction()` 自动压缩**：挂载该能力，让 SDK 在上下文压力需要时自动压缩。
2. **使用 `StaticCompactionPolicy` 基于阈值压缩**：为希望上下文大小行为更可预测的环境设置明确阈值。
3. **使用 `OpenAIResponsesCompactionSession.run_compaction({"force": True})` 强制检查点压缩**：在应用定义的阶段边界处压缩，例如在一个主要审阅阶段之后、下一批证据之前。

本 notebook 使用强制检查点，因为合成数据集刻意做得很小。在生产环境中，自动压缩通常是最简单的起点，而当你想要更严格的运维策略时，基于阈值的压缩会很有用。

> **最佳实践**
>
> - 在有意义的工作流边界处压缩，而不是每轮之后都压缩。
> - 保留足够的工作状态，让下一阶段仍然说得通。
> - 把引用的事实保存在生成的产物中，而不仅仅保存在压缩后的对话状态里。

In [ ]:
from agents.sandbox.capabilities import Compaction, StaticCompactionPolicy


def build_compaction_agent(*, demo_threshold: int | None = None) -> SandboxAgent:
    if demo_threshold is None:
        compaction = Compaction()
    else:
        compaction = Compaction(policy=StaticCompactionPolicy(threshold=demo_threshold))

    return SandboxAgent(
        name="Evidence Review Agent with Compaction",
        model=MODEL,
        instructions=(
            BASELINE_INSTRUCTIONS
            + "\n\nWhen context is compacted, preserve the current batch, cited facts, open "
            "questions, artifact paths, and unresolved reviewer concerns."
        ),
        default_manifest=build_evidence_manifest(),
        capabilities=[
            Filesystem(),
            Shell(),
            compaction,
        ],
    )


compaction_agent = build_compaction_agent()
threshold_compaction_agent = build_compaction_agent(demo_threshold=8_000)
print({
    "automatic": [type(capability).__name__ for capability in compaction_agent.capabilities],
    "threshold_policy": [type(capability).__name__ for capability in threshold_compaction_agent.capabilities],
})


### 压缩如何被触发

挂载 `Compaction()` 能力后，当活跃上下文增长到足够大时，服务端压缩就有资格运行。这是当前的默认行为：挂载能力，让 SDK 管理上下文压力。

对于小型教程，自动压缩可能很难观察到，因为运行可能永远接近不了模型的上下文上限。设置更低的 `StaticCompactionPolicy` 会有帮助，但它仍然依赖渲染后的上下文越过阈值。

对于小型证据集，强制检查点是最清晰的操作模式。`OpenAIResponsesCompactionSession` 包装器会存储会话历史，并允许应用在阶段边界调用 `run_compaction({"force": True})`。这样无需膨胀证据集就能让压缩变得可见。


In [ ]:
from agents.memory import OpenAIResponsesCompactionSession, SQLiteSession


def build_compaction_session() -> OpenAIResponsesCompactionSession:
    underlying = SQLiteSession("evidence_review_session.sqlite")
    return OpenAIResponsesCompactionSession(
        session_id="evidence-review-demo",
        underlying_session=underlying,
        model=COMPACTION_MODEL,
        compaction_mode="input",
    )


async def force_compaction_checkpoint(session: OpenAIResponsesCompactionSession) -> dict[str, int]:
    items_before = await session.get_items()
    await session.run_compaction({"force": True, "compaction_mode": "input"})
    items_after = await session.get_items()
    return {"items_before": len(items_before), "items_after": len(items_after)}


print("Compaction session helper defined. The final run uses it to show an explicit phase checkpoint.")


## 第 3 步：挂载记忆

记忆用于跨运行复用。在本例中，记忆应当捕获**工作流经验**，而不是调查事实。

适合写入记忆的内容包括：

- 审阅基于文件的证据工作区时先看清单。
- 在备忘录中保留不确定性，而不是猜测。
- 当后续证据缩小了早先的假设时，让早先的假设保持可见。

不适合写入记忆的内容包括：

- “Northwind Logistics 违反了政策。”
- “ACME 的 Finance Ops 流程存在缺陷。”
- 任何应当写进备忘录的案件相关结论。

> **最佳实践**
>
> - 把记忆用于稳定的流程经验和用户偏好。
> - 把案件相关的事实保存在备忘录等经审阅的产物中。
> - 在后续运行依赖生成的记忆之前，先检查它。

In [ ]:
from agents.sandbox import MemoryGenerateConfig
from agents.sandbox.capabilities import Memory

MEMORY_GENERATION_PROMPT = """
Store reusable workflow lessons only.
Do not store ACME-specific compliance findings, document facts, evidence citations,
or memo conclusions. Those belong in outputs/compliance_review_memo.md.
Memory should help future evidence-review workflows behave better; it should not
become a second investigation record.
""".strip()


def workflow_memory() -> Memory:
    return Memory(
        generate=MemoryGenerateConfig(
            extra_prompt=MEMORY_GENERATION_PROMPT,
        )
    )


def build_memory_agent() -> SandboxAgent:
    return SandboxAgent(
        name="Evidence Review Agent with Memory",
        model=MODEL,
        instructions=BASELINE_INSTRUCTIONS,
        default_manifest=build_evidence_manifest(),
        capabilities=[
            Filesystem(),
            Shell(),
            workflow_memory(),
        ],
    )


memory_agent = build_memory_agent()
print([type(capability).__name__ for capability in memory_agent.capabilities])


## 第 4 步：同时启用压缩与记忆运行

现在把各部分组合起来：

- `Filesystem()` 和 `Shell()` 让智能体在证据工作区中导航。
- `Compaction()` 让活跃的审阅在上下文增长时仍能继续。
- `Memory()` 在运行结束后捕获可复用的工作流经验。
- 最终备忘录始终是调查产物。

下面的任务要求智能体审阅合成证据、撰写备忘录，然后读回备忘录，验证它是否保留了要求的结构和不确定性。

In [ ]:
FINAL_REVIEW_TASK = """
Review all three document batches in order.

For each batch:
1. Read the manifest and relevant documents.
2. Preserve cited findings and uncertainty in your working notes and final memo.
3. Preserve any superseded or narrowed assumption instead of silently deleting it.

After Batch 3, write `outputs/compliance_review_memo.md` with:
- executive summary,
- cited findings table,
- open questions,
- recommended next steps for the reviewer.

Reviewer preference for future runs: keep the memo concise, preserve uncertainty
instead of guessing, and separate reusable workflow lessons from document-specific
compliance findings.
""".strip()


def build_reliable_evidence_agent() -> SandboxAgent:
    return SandboxAgent(
        name="Reliable Evidence Review Agent",
        model=MODEL,
        instructions=(
            BASELINE_INSTRUCTIONS
            + "\n\nUse compaction as working context. Use SDK memory for reusable "
            "workflow lessons across runs. Do not treat memory as the system of record "
            "for ACME-specific findings; those belong in the cited memo artifact."
        ),
        default_manifest=build_evidence_manifest(),
        capabilities=[
            Filesystem(),
            Shell(),
            Compaction(),
            workflow_memory(),
        ],
    )


reliable_agent = build_reliable_evidence_agent()
print([type(capability).__name__ for capability in reliable_agent.capabilities])


In [ ]:
EXAMPLE_OUTPUT = """
# Compliance Review Memo

## Executive Summary

The current record supports a control-gap finding for the Northwind Logistics
vendor exception, not a final conclusion that policy was intentionally violated.
The strongest evidence is that ACME required two approvals and Finance Ops
logging for payment exceptions, while the Northwind exception appears to have
been released before Finance Ops approval was complete.

Later evidence narrows the initial concern. The record no longer points only to
an undocumented verbal exception; it now points to a specific process weakness:
Legal approval may have been obtained, but Finance Ops review and exception-log
reconciliation remained incomplete at the time of release.

## Cited Findings

| Finding | Support | Status |
|---|---|---|
| Payments over $50,000 required two approvals, and exceptions had to be logged with Finance Ops. | ACME-B1-001 | Supported |
| The Northwind Logistics exception was approved verbally, but the initial note does not show a Finance Ops log entry. | ACME-B1-002 | Supported |
| Internal Audit treated missing exception logs as a control gap until resolved. | ACME-B2-001 | Supported |
| The approval thread indicates Legal approved the exception, but Finance Ops approval was still pending when payment was released. | ACME-B2-002 | Supported |
| The remediation plan requires Finance Ops to reconcile verbal vendor exceptions from Q4 and add retrospective control attestations. | ACME-B3-001 | Supported |

## Open Questions

- Was Finance Ops approval completed after the payment release, and if so, when?
- How many other Q4 verbal vendor exceptions lack retrospective attestations?
- Did any compensating control apply to the Northwind payment before remediation began?

## Recommended Next Steps

1. Reconcile Northwind Logistics against the Finance Ops exception log and payment-release timestamp.
2. Pull the full Q4 population of verbal vendor exceptions into the same review workflow.
3. Classify the issue as a control gap unless later evidence shows timely Finance Ops approval or an approved compensating control.
""".strip()

if not RUN_AGENT:
    print("RUN_AGENT is False, so this cell shows an example memo shape rather than running the model.\n")
    print(EXAMPLE_OUTPUT)


## 检查生成的产物

智能体的最终回复固然有用，但当你检查沙箱运行产生的文件时，这套可靠性模式会更加清晰。本节把通常隐藏的状态展示出来：

- 面向审阅人的备忘录 `outputs/compliance_review_memo.md`，
- 生成的 SDK 记忆文件，例如 `memories/MEMORY.md` 和 `memories/memory_summary.md`，
- 运行产生的工作区文件，包括会话日志。

生成的记忆产物不是合规备忘录，不应被当作调查事实。它是可复用的工作流记忆。`Task Group` 标题是记忆系统自己的分组标签，记忆生成器通过 `MemoryGenerateConfig.extra_prompt` 被引导为存储工作流经验，而不是 ACME 相关的发现。

如果 `RUN_AGENT = False`，本节会展示预期的输出形态，而不是真实的沙箱产物。


In [ ]:
try:
    from IPython.display import Markdown, display
except ImportError:
    Markdown = None
    display = None

memo_text = final_run.get("memo") if "final_run" in globals() else None
workspace_files = final_run.get("workspace_files") if "final_run" in globals() else None
memory_artifacts = final_run.get("memory_artifacts", {}) if "final_run" in globals() else {}
compaction_checkpoint = final_run.get("compaction_checkpoint") if "final_run" in globals() else None

print("Generated workspace files:")
print(workspace_files or "No generated workspace files were captured.")

print("\nMemory and compaction configuration:")
print({
    "final_agent_capabilities": [type(capability).__name__ for capability in reliable_agent.capabilities],
    "sandbox_compaction": "Compaction() is attached to the final agent for automatic context management",
    "sandbox_memory": "Memory(generate=MemoryGenerateConfig(...)) is attached to the final agent",
    "memory_write_policy": "Store reusable workflow lessons, not ACME-specific compliance findings",
    "forced_checkpoint": "OpenAIResponsesCompactionSession.run_compaction({force: True}) after the final review",
    "compaction_model_for_checkpoint": COMPACTION_MODEL,
    "checkpoint_result": compaction_checkpoint,
})

if memory_artifacts:
    for path, text in memory_artifacts.items():
        heading = f"### Generated SDK memory artifact: `{path}`\n\n"
        explanation = (
            "This block is generated by the SDK `Memory()` primitive. It is reusable "
            "workflow memory, not the compliance memo and not the investigation system "
            "of record.\n\n"
            "```text\n----- BEGIN GENERATED SDK MEMORY ARTIFACT -----\n"
        )
        closing = "\n----- END GENERATED SDK MEMORY ARTIFACT -----\n```"
        if display is not None and Markdown is not None:
            display(Markdown(heading + explanation + text + closing))
        else:
            print(f"\nGenerated SDK memory artifact: {path}\n")
            print("----- BEGIN GENERATED SDK MEMORY ARTIFACT -----")
            print(text)
            print("----- END GENERATED SDK MEMORY ARTIFACT -----")
else:
    print("\nNo generated memory artifacts were captured. Set RUN_AGENT = True and rerun the final workflow.")

if memo_text:
    if display is not None and Markdown is not None:
        display(Markdown("## Generated memo: `outputs/compliance_review_memo.md`\n\n" + memo_text))
    else:
        print("\nGenerated memo:\n")
        print(memo_text)
else:
    print("\nNo memo was captured. Set RUN_AGENT = True and rerun the final workflow.")


**常见误区**

不要把 `Memory()` 当作未经审阅的事实数据库。

记忆应当帮助下一次运行记住如何工作，而不应成为影子合规记录。如果某个结论重要，就把它连同引用写进经审阅的产物中。

## 结论

你现在已经拥有构建可靠的长时间运行智能体工作流所需的基础组件：

- 用于受控文件访问的沙箱工作区。
- 帮助智能体在文档间定位的清单。
- 应对有限上下文窗口的压缩。
- 存放可复用工作流经验的记忆。
- 作为经审阅调查产物的生成备忘录。

核心设计选择是职责分离：上下文帮助智能体工作，记忆帮助后续智能体工作得更好，经审阅的产物则承载人们将依赖的事实。